# Imeplementación de modelos de Redes Neuronales Profundas

En este archivo, voy a poner en práctica lo que hemos ido aprendiendo a lo largo de la asignatura tanto en la parte de toería como en la parte de práctica. La idea es crear una CNN la cual sea capaz de analizar fotos de paciente con Pneumonia y sin está; esto nos va a permitir entrenar el modelo ver cómo actúa y si es capaz de funcionar de manera correcta. 

**Realizado por Rodrigo Gálvez Travalja**

## Importación de las librerias 

In [3]:
%pip install torch torchvision 
%pip install kagglehub
%pip install matplotlib
%pip install pandas

import kagglehub    
import pandas as pd
import heapq
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
%pip install torchvision 
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.transforms.functional import to_pil_image
from PIL import Image


import torch
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device name:", torch.cuda.get_device_name(0))
    print("cuda runtime version:", torch.version.cuda)

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.7 MB/s eta 0:00:00
  Using cached contourpy-1.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (117 kB)
  Using cached kiwisolver-1.5.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 5.8 MB/s eta 0:00:0000:0100:01
Using cached contourpy-1.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (362 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.62.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (5.0 

# Comprobación de GPU y configuración de `device`
# Esta celda verifica si PyTorch detecta CUDA y crea la variable `device`.


In [4]:
import torch

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device name:", torch.cuda.get_device_name(0))
    print("cuda runtime version:", torch.version.cuda)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)


def move_to_device(batch, device):
    """Mueve tensores dentro de `batch` al `device`.
    Soporta `torch.Tensor`, tuplas/listas y diccionarios.
    """
    if isinstance(batch, (list, tuple)):
        return type(batch)(move_to_device(x, device) for x in batch)
    if isinstance(batch, dict):
        return {k: move_to_device(v, device) for k, v in batch.items()}
    try:
        return batch.to(device)
    except Exception:
        return batch


torch version: 2.11.0+cu130
cuda available: True
device name: NVIDIA GeForce RTX 4060 Ti
cuda runtime version: 13.0
using device: cuda


## Descarga de los archivos de Kaggle 

In [ ]:
ruta = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
archivos = os.listdir(ruta)
# Mostrar el contenido de la ruta descargada
for archivo in archivos:
    print(archivo)


ruta_100x100 = os.path.join(ruta, 'chest_xray', 'train', 'NORMAL')
os.chdir(ruta_100x100)
print(f"Directorio actual: {os.getcwd()}")

# Listar el contenido del directorio correctamente
for carpeta in os.listdir(ruta_100x100):
    ruta_carpeta = os.path.join(ruta_100x100, carpeta)
    print(f"Carpeta: {carpeta}")
    if os.path.isdir(ruta_carpeta):  # Verificar si es un directorio
        print(os.listdir(ruta_carpeta))
# --- Descargar dataset desde KaggleHub ---
directorio = os.path.join(ruta, 'chest_xray')
RUTA_ENTRENAMIENTO = os.path.join(directorio, "train")
RUTA_PRUEBA = os.path.join(directorio, "Test")  
        

100%|██████████| 2.29G/2.29G [01:31<00:00, 27.0MB/s]

Extracting files...


chest_xray
Directorio actual: /Users/rodrigo/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray/train/NORMAL
Carpeta: NORMAL2-IM-0927-0001.jpeg
Carpeta: NORMAL2-IM-1056-0001.jpeg
Carpeta: IM-0427-0001.jpeg
Carpeta: NORMAL2-IM-1260-0001.jpeg
Carpeta: IM-0656-0001-0001.jpeg
Carpeta: IM-0561-0001.jpeg
Carpeta: NORMAL2-IM-1110-0001.jpeg
Carpeta: IM-0757-0001.jpeg
Carpeta: NORMAL2-IM-1326-0001.jpeg
Carpeta: NORMAL2-IM-0736-0001.jpeg
Carpeta: NORMAL2-IM-0500-0001.jpeg
Carpeta: NORMAL2-IM-0393-0001.jpeg
Carpeta: NORMAL2-IM-0994-0001.jpeg
Carpeta: IM-0207-0001.jpeg
Carpeta: IM-0494-0001.jpeg
Carpeta: IM-0177-0001.jpeg
Carpeta: IM-0388-0001.jpeg
Carpeta: IM-0341-0001.jpeg
Carpeta: IM-0355-0001.jpeg
Carpeta: IM-0449-0001.jpeg
Carpeta: IM-0480-0001.jpeg
Carpeta: NORMAL2-IM-1038-0001.jpeg
Carpeta: NORMAL2-IM-1348-0001.jpeg
Carpeta: IM-0739-0001.jpeg
Carpeta: IM-0213-0001.jpeg
Carpeta: NORMAL2-IM-0452-0001.jpeg
Carpeta: NORMAL2-IM-0980-0001.jpeg
Carpeta: NORMAL2-